In [1]:
# Week 5 Spark Assignment
# Environment Setup

!pip install pyspark

In [2]:
# Spark Initialization
# Required for all questions

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week5SparkAssignment") \
    .getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


# =====================================================
# Q1
# What are the limitations of MapReduce?
# =====================================================

Q1 Answer:

Limitations of MapReduce:
1. Disk-based processing.
2. Slower execution speed.
3. High latency.
4. Complex programming model.
5. Poor support for iterative algorithms.

Advantages of Spark:
1. In-memory computing.
2. Faster execution.
3. Easy APIs.
4. Better machine learning support.
5. Real-time processing capabilities

# =====================================================
# Q2
# Explain In-Memory Computing
# =====================================================

Q2 Answer:

Spark stores intermediate data in memory (RAM)
instead of repeatedly writing data to disk.

Benefits:
- Faster processing
- Reduced disk I/O
- Better performance for iterative machine learning
- Efficient data analytics

In [3]:
# =====================================================
# STEP 3: CREATE SAMPLE DATASET
# Covers Q3 to Q15 Requirements
# =====================================================

import pandas as pd

data = {
    "user_id":[101,101,102,103,104,105,106,107,108,109,110],
    "transaction_date":[
        "2025-01-01","2025-01-01","2025-01-02","2025-01-03",
        "2025-01-04","2025-01-05","2025-01-06","2025-01-07",
        "2025-01-08","2025-01-09","2025-01-10"
    ],
    "region":[
        "West","West","West","East","West",
        "West","South","West","North","West","West"
    ],
    "product_category":[
        "Electronics","Electronics","Furniture",
        "Electronics","Clothing","Electronics",
        "Furniture","Electronics","Clothing",
        "Furniture","Electronics"
    ],
    "sale_amount":[500,500,300,700,200,600,400,750,450,900,650],
    "city":[
        "Delhi","Delhi","Mumbai","Kolkata",
        "Delhi","Delhi","Chennai","Delhi",
        "Jaipur","Mumbai","Delhi"
    ],
    "age":[25,25,29,17,30,28,22,26,31,24,19],
    "subscription":[
        "Premium","Premium","Premium","Basic",
        "Premium","Premium","Basic","Premium",
        "Basic","Premium","Premium"
    ],
    "status":[None,None,"Active",None,None,
              "Active",None,None,"Active",None,"Active"],
    "price":[100,100,None,150,80,200,120,250,None,300,180],
    "store_id":[1,1,2,1,2,1,3,1,2,2,1],
    "email":[
        "test1@gmail.com","test1@gmail.com",
        "test2@gmail.com",None,
        "test4@gmail.com","test5@gmail.com",
        "test6@gmail.com","test7@gmail.com",
        "test8@gmail.com",None,
        "test10@gmail.com"
    ],
    "username":[
        "user1","user1","user2","user3",
        "", "user5","user6","user7",
        "user8","user9",""
    ],
    "raw_timestamp":[
        "2025-01-01 10:00:00",
        "2025-01-01 10:00:00",
        "2025-01-02 11:00:00",
        "2025-01-03 12:00:00",
        "2025-01-04 09:00:00",
        "2025-01-05 08:00:00",
        "2025-01-06 07:00:00",
        "2025-01-07 14:00:00",
        "2025-01-08 15:00:00",
        "2025-01-09 16:00:00",
        "2025-01-10 17:00:00"
    ]
}

pdf = pd.DataFrame(data)

pdf.to_csv("sample_data.csv", index=False)

print("Dataset Created Successfully")

Dataset Created Successfully


In [4]:
# =====================================================
# STEP 4: LOAD DATASET INTO SPARK DATAFRAME
# Objective: Spark DataFrame Concepts
# =====================================================

df = spark.read.csv(
    "sample_data.csv",
    header=True,
    inferSchema=True
)

df.show()

print("Total Records:", df.count())

+-------+----------------+------+----------------+-----------+-------+---+------------+------+-----+--------+----------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   city|age|subscription|status|price|store_id|           email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+-------+---+------------+------+-----+--------+----------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|  Delhi| 25|     Premium|  NULL|100.0|       1| test1@gmail.com|   user1|2025-01-01 10:00:00|
|    101|      2025-01-01|  West|     Electronics|        500|  Delhi| 25|     Premium|  NULL|100.0|       1| test1@gmail.com|   user1|2025-01-01 10:00:00|
|    102|      2025-01-02|  West|       Furniture|        300| Mumbai| 29|     Premium|Active| NULL|       2| test2@gmail.com|   user2|2025-01-02 11:00:00|
|    103|      2025-01-03|  East|     Electronics|        700|Ko

In [5]:
# =====================================================
# Q3
# Remove duplicate rows based on:
# user_id and transaction_date
# =====================================================

df_q3 = df.dropDuplicates(["user_id", "transaction_date"])

print("Original Records:", df.count())
print("Records After Removing Duplicates:", df_q3.count())

df_q3.show()

Original Records: 11
Records After Removing Duplicates: 10
+-------+----------------+------+----------------+-----------+-------+---+------------+------+-----+--------+----------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   city|age|subscription|status|price|store_id|           email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+-------+---+------------+------+-----+--------+----------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|  Delhi| 25|     Premium|  NULL|100.0|       1| test1@gmail.com|   user1|2025-01-01 10:00:00|
|    102|      2025-01-02|  West|       Furniture|        300| Mumbai| 29|     Premium|Active| NULL|       2| test2@gmail.com|   user2|2025-01-02 11:00:00|
|    103|      2025-01-03|  East|     Electronics|        700|Kolkata| 17|       Basic|  NULL|150.0|       1|            NULL|   user3|2025-01-03 12:00:00|
|    

In [6]:
# =====================================================
# Q4
# Filter Region = West
# Group by Product Category
# Find Average Sale Amount
# =====================================================

df_q4 = (
    df.filter(df.region == "West")
      .groupBy("product_category")
      .avg("sale_amount")
)

df_q4.show()

+----------------+----------------+
|product_category|avg(sale_amount)|
+----------------+----------------+
|     Electronics|           600.0|
|        Clothing|           200.0|
|       Furniture|           600.0|
+----------------+----------------+



In [7]:
# =====================================================
# Q5
# Handle Null Values
# Difference:
#
# na.drop() -> removes rows containing null values
# na.fill() -> replaces null values
# =====================================================

df_q5 = df.na.fill({"status": "Unknown"})

df_q5.select("user_id", "status").show()

+-------+-------+
|user_id| status|
+-------+-------+
|    101|Unknown|
|    101|Unknown|
|    102| Active|
|    103|Unknown|
|    104|Unknown|
|    105| Active|
|    106|Unknown|
|    107|Unknown|
|    108| Active|
|    109|Unknown|
|    110| Active|
+-------+-------+



In [8]:
# =====================================================
# Q6
# Count Records Per City
# Show Only Cities Having Count > 1
# (Using >1 because sample dataset is small)
# =====================================================

from pyspark.sql.functions import count

df_q6 = (
    df.groupBy("city")
      .agg(count("*").alias("total_records"))
      .filter("total_records > 1")
)

df_q6.show()

+------+-------------+
|  city|total_records|
+------+-------------+
|Mumbai|            2|
| Delhi|            6|
+------+-------------+



# =====================================================
# Q7
# DataFrame Immutability
# =====================================================

Q7 Answer:

Spark DataFrames are immutable.

This means operations such as:

- drop()
- withColumnRenamed()
- filter()

do not modify the original DataFrame.

Instead, they create and return a new DataFrame.

Example:

new_df = df.drop("column_name")

Original DataFrame remains unchanged.

In [9]:
# =====================================================
# Q8
# Filter rows where:
# Age is between 18 and 30 (inclusive)
# AND Subscription = Premium
# =====================================================

df_q8 = df.filter(
    (df.age >= 18) &
    (df.age <= 30) &
    (df.subscription == "Premium")
)

df_q8.show()

+-------+----------------+------+----------------+-----------+------+---+------------+------+-----+--------+----------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|  city|age|subscription|status|price|store_id|           email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+------+---+------------+------+-----+--------+----------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500| Delhi| 25|     Premium|  NULL|100.0|       1| test1@gmail.com|   user1|2025-01-01 10:00:00|
|    101|      2025-01-01|  West|     Electronics|        500| Delhi| 25|     Premium|  NULL|100.0|       1| test1@gmail.com|   user1|2025-01-01 10:00:00|
|    102|      2025-01-02|  West|       Furniture|        300|Mumbai| 29|     Premium|Active| NULL|       2| test2@gmail.com|   user2|2025-01-02 11:00:00|
|    104|      2025-01-04|  West|        Clothing|        200| Delhi| 

# =====================================================
# Q9
# Why handle null values before aggregations?
# =====================================================


Q9 Answer:

Null values should be handled before performing
aggregations like sum() and avg().

Reasons:

1. Incorrect calculations may occur.
2. Missing values can affect averages.
3. Business insights become unreliable.
4. Data quality improves after cleaning.

Therefore null handling is an important
data cleaning step before aggregation.

In [10]:
# =====================================================
# Q10
# Convert raw_timestamp to TimestampType
# Rename column to event_time
# =====================================================

from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

df_q10 = (
    df.withColumn(
        "event_time",
        col("raw_timestamp").cast(TimestampType())
    )
    .drop("raw_timestamp")
)

df_q10.printSchema()

df_q10.show()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

+-------+----------------+------+----------------+-----------+-------+---+------------+------+-----+--------+----------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   city|age|subscription|status|price|store_id|           email|username|         event_time|
+-------+----------------+------+----------------+-----------+-------+---+------------+------+-----+--------+---

# =====================================================
# Q11
# Explain Shuffle Process
# =====================================================


Q11 Answer:

Shuffle is the process of redistributing data
across partitions during operations such as:

- groupBy()
- join()
- reduceByKey()

Why is it called a Wide Transformation?

Because data moves between partitions and
different executors in the cluster.

Characteristics:

1. Network communication occurs.
2. More expensive than narrow transformations.
3. Required for grouping and aggregation.
4. Can impact performance if data is large.

In [11]:
# =====================================================
# Objective Demonstration
# Wide Transformation Example
# groupBy causes shuffle
# =====================================================

df.groupBy("region").sum("sale_amount").show()

+------+----------------+
|region|sum(sale_amount)|
+------+----------------+
| South|             400|
|  East|             700|
|  West|            4400|
| North|             450|
+------+----------------+



In [12]:
# =====================================================
# Q12
# Remove rows where:
# email is NULL
# OR
# username is empty
# =====================================================

df_q12 = df.filter(
    (df.email.isNotNull()) &
    (df.username != "")
)

print("Original Records:", df.count())
print("Cleaned Records:", df_q12.count())

df_q12.show()

Original Records: 11
Cleaned Records: 7
+-------+----------------+------+----------------+-----------+-------+---+------------+------+-----+--------+---------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   city|age|subscription|status|price|store_id|          email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+-------+---+------------+------+-----+--------+---------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|  Delhi| 25|     Premium|  NULL|100.0|       1|test1@gmail.com|   user1|2025-01-01 10:00:00|
|    101|      2025-01-01|  West|     Electronics|        500|  Delhi| 25|     Premium|  NULL|100.0|       1|test1@gmail.com|   user1|2025-01-01 10:00:00|
|    102|      2025-01-02|  West|       Furniture|        300| Mumbai| 29|     Premium|Active| NULL|       2|test2@gmail.com|   user2|2025-01-02 11:00:00|
|    105|      2025-01-05|  We

In [13]:
# =====================================================
# Q13
# Calculate Min, Max and Mean of Price
# =====================================================

from pyspark.sql.functions import min, max, avg

df.agg(
    min("price").alias("Minimum_Price"),
    max("price").alias("Maximum_Price"),
    avg("price").alias("Average_Price")
).show()

+-------------+-------------+------------------+
|Minimum_Price|Maximum_Price|     Average_Price|
+-------------+-------------+------------------+
|         80.0|        300.0|164.44444444444446|
+-------------+-------------+------------------+



# =====================================================
# Q14
# Risk of inferSchema=True with messy data
# =====================================================

Q14 Answer:

Using inferSchema=True on inconsistent datasets
can create incorrect data types.

Example:

Some rows:
2025-01-01

Other rows:
01/01/2025

Spark may:

1. Infer wrong datatype.
2. Convert values to NULL.
3. Cause schema mismatch.
4. Produce incorrect analysis results.

Therefore schema should be validated
when working with messy datasets.

In [14]:
# =====================================================
# Q15
# Complete Data Processing Pipeline
#
# 1. Remove duplicates
# 2. Fill null prices with 0
# 3. Group by store_id
# 4. Calculate total revenue
# =====================================================

from pyspark.sql.functions import sum

df_q15 = (
    df
    .dropDuplicates(["user_id", "transaction_date"])
    .na.fill({"price": 0})
    .groupBy("store_id")
    .agg(
        sum("price").alias("total_revenue")
    )
)

df_q15.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|       1|        880.0|
|       3|        120.0|
|       2|        380.0|
+--------+-------------+



# =====================================================
# FINAL INSIGHTS
# =====================================================

print("""
Assignment Insights:

1. Spark DataFrames are immutable.
2. Duplicate records were removed successfully.
3. Null values were handled using fill().
4. Filtering operations improved data quality.
5. GroupBy and Aggregations generated useful insights.
6. Schema modification converted timestamp columns.
7. Wide transformations cause shuffle operations.
8. A complete ETL-style pipeline was created.
""")